# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [9]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [6]:
# Tran Test Split

features = df.drop(columns=["Transported"])
target = df["Transported"]

x_train, x_test, y_train, y_test = train_test_split(features, target, test_size = 0.2, random_state = 42)

# Cabin simplification

x_train["Cabin"] = x_train["Cabin"].str[0]
x_test["Cabin"] = x_test["Cabin"].str[0]

# Drop PassengerId and Name columns

x_train = x_train.drop(columns=["PassengerId", "Name"])
x_test = x_test.drop(columns=["PassengerId", "Name"])

# Encoding and Null values info

df.info()

# Encoding

cat_cols = ["HomePlanet", "Cabin", "Destination"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

train_encoded = encoder.fit_transform(x_train[cat_cols])

test_encoded = encoder.transform(x_test[cat_cols])

train_encoded_df = pd.DataFrame(train_encoded, columns=encoder.get_feature_names_out(cat_cols))
test_encoded_df = pd.DataFrame(test_encoded, columns=encoder.get_feature_names_out(cat_cols))

# Drop categorical features after encoding

x_train = x_train.drop(columns=cat_cols)
x_test = x_test.drop(columns=cat_cols)

# Concadenating Encoded data

x_train = pd.concat([x_train.reset_index(drop=True), train_encoded_df.reset_index(drop=True)], axis=1)
x_test = pd.concat([x_test.reset_index(drop=True), test_encoded_df.reset_index(drop=True)], axis=1)

# Fill NaN values fro Boolean features

x_train["CryoSleep"] = x_train["CryoSleep"].fillna(False).astype(int)
x_test["CryoSleep"] = x_test["CryoSleep"].fillna(False).astype(int)

x_train["VIP"] = x_train["VIP"].fillna(False).astype(int)
x_test["VIP"] = x_test["VIP"].fillna(False).astype(int)

# Changing type for boolean features

x_train["CryoSleep"] = x_train["CryoSleep"].astype(int)
x_test["CryoSleep"] = x_test["CryoSleep"].astype(int)

x_train["VIP"] = x_train["VIP"].astype(int)
x_test["VIP"] = x_test["VIP"].astype(int)

y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Filling NaN values with the mean

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["Age"]])
x_train["Age"] =imputer.transform(x_train[["Age"]])
x_test["Age"] =imputer.transform(x_test[["Age"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["RoomService"]])
x_train["RoomService"] =imputer.transform(x_train[["RoomService"]])
x_test["RoomService"] =imputer.transform(x_test[["RoomService"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["FoodCourt"]])
x_train["FoodCourt"] =imputer.transform(x_train[["FoodCourt"]])
x_test["FoodCourt"] =imputer.transform(x_test[["FoodCourt"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["ShoppingMall"]])
x_train["ShoppingMall"] =imputer.transform(x_train[["ShoppingMall"]])
x_test["ShoppingMall"] =imputer.transform(x_test[["ShoppingMall"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["Spa"]])
x_train["Spa"] =imputer.transform(x_train[["Spa"]])
x_test["Spa"] =imputer.transform(x_test[["Spa"]])

imputer = SimpleImputer(strategy= "mean")
imputer.fit(x_train[["VRDeck"]])
x_train["VRDeck"] =imputer.transform(x_train[["VRDeck"]])
x_test["VRDeck"] =imputer.transform(x_test[["VRDeck"]])

# Feature selection

columns_to_drop = ["Destination_TRAPPIST-1e", "HomePlanet_Earth", "HomePlanet_Europa", "Cabin_T", "Destination_PSO J318.5-22", "Cabin_A"]

x_train = x_train.drop(columns=columns_to_drop, errors="ignore")
x_test = x_test.drop(columns=columns_to_drop, errors="ignore")

# Feature scaling: Standarization

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)



<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

- Evaluate your model

In [7]:
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(x_train_scaled, y_train)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, gb_clf.predict(x_test_scaled)))

Gradient Boosting Accuracy: 0.7901092581943646


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [12]:
gb_clf = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}

- Run Grid Search

In [13]:
grid_search = GridSearchCV(
    estimator=gb_clf,
    param_grid=param_grid,
    cv=5,         
    scoring='accuracy',
    n_jobs=-1       
)

grid_search.fit(x_train_scaled, y_train)

print("Hyperparameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Hyperparameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50}
Best score: 0.8035679522521451


- Evaluate your model

In [14]:
best_model = grid_search.best_estimator_
test_accuracy = accuracy_score(y_test, best_model.predict(x_test_scaled))
print("Accuracy:", test_accuracy)

Accuracy: 0.7901092581943646
